# EDA of combined energy demand and weather data

## Load and combine dataset

In [ ]:
import pandas as pd

df_energy = pd.read_csv("../data/energy_demand_de_2019_2025.csv", parse_dates=['DateUTC'])
df_weather_germany = pd.read_csv("../data/weather_data_mean_cities_2019_2025.csv", parse_dates=['DateUTC'])

# combine energy demand data with weather data, and save the combined dataset for future use
df_energy_weather = pd.concat([df_energy, df_weather_germany], axis=1)
df_energy_weather = df_energy_weather.loc[:,~df_energy_weather.columns.duplicated()]

df_energy_weather = df_energy_weather.sort_values('DateUTC').reset_index(drop=True)

df_energy_weather.head()


In [ ]:
display(df_energy_weather.describe())
display(df_energy_weather.info())

In [ ]:
# feature engineering: create new features based on existing ones, such as rolling averages, lagged variables, or interaction terms
df_energy_weather['apparent_temperature_rolling_mean_24h'] = df_energy_weather['apparent_temperature'].shift(1).rolling(window=24).mean()
df_energy_weather['apparent_temperature_lag_24h'] = df_energy_weather['apparent_temperature'].shift(24)

# add rolling average and lagged varirable for shortwave_radiation_0m
df_energy_weather['shortwave_radiation_0m_rolling_mean_24h'] = df_energy_weather['shortwave_radiation'].shift(1).rolling(window=24).mean()
df_energy_weather['shortwave_radiation_0m_lag_24h'] =   df_energy_weather['shortwave_radiation'].shift(24)

# add heating degree days (HDD) and cooling degree days (CDD) features
base_temperature_heating = 15  # base temperature for heating degree days
base_temperature_cooling = 25  # base temperature for cooling degree days
df_energy_weather['heating_degree'] = df_energy_weather['apparent_temperature'].apply(lambda x: max(0, base_temperature_heating - x))  # HDD is calculated as the difference between a base temperature (e.g., 18°C) and the actual temperature, but only if the actual temperature is below the base temperature
df_energy_weather['cooling_degree'] = df_energy_weather['apparent_temperature'].apply(lambda x: max(0, x - base_temperature_cooling))  # CDD is calculated as the difference between the actual temperature and a base temperature (e.g., 25°C), but only if the actual temperature is above the base temperature

# add pandemic feature
pandemic_start = pd.to_datetime('2020-03-01')
pandemic_end = pd.to_datetime('2021-12-31')
df_energy_weather['is_pandemic_time'] = df_energy_weather['DateUTC'].apply(lambda x: 1 if (x >= pandemic_start) and (x <= pandemic_end) else 0)

# add lagged features for energy demand (shifted by 24 hours, 168 hours (1 week) to capture daily, weekly, and yearly patterns)
df_energy_weather['EnergyDemand_lag_24h'] = df_energy_weather['EnergyDemand'].shift(24)   # 1 day
df_energy_weather['EnergyDemand_lag_168h'] = df_energy_weather['EnergyDemand'].shift(168)   # 1 week
df_energy_weather['EnergyDemand_lag_8760h'] = df_energy_weather['EnergyDemand'].shift(8760) # 1 year, removed after feature importance analysis showed it was not useful

# rolling mean of past demand (shift first to avoid leakage)
df_energy_weather['EnergyDemand_rolling_mean_24h'] = df_energy_weather['EnergyDemand'].shift(1).rolling(24).mean()   # daily pattern
df_energy_weather['EnergyDemand_rolling_mean_168h'] = df_energy_weather['EnergyDemand'].shift(1).rolling(168).mean() # weekly pattern
df_energy_weather['EnergyDemand_rolling_mean_8760h'] = df_energy_weather['EnergyDemand'].shift(1).rolling(8760).mean() # yearly pattern, removed after feature importance analysis showed it was not useful

# drop rows with missing values
df_energy_weather = df_energy_weather.dropna()

#display(df_energy_weather.isna().sum())

df_energy_weather.to_csv("../data/energy_weather_2019_2025.csv", index=False)


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# change the size of the plot
plt.figure(figsize=(20, 18))
energy_weather_corr = df_energy_weather.drop('DateUTC', axis=1).corr()
sns.heatmap(energy_weather_corr, annot=True, cmap='coolwarm')
plt.title('Correlation Heatmap of Energy and Weather Variables') 
plt.show()

In [ ]:
# plot energy demand histogram
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

import matplotlib.pyplot as plt
plt.rcParams.update({
    'axes.grid':      True,
    'grid.color':     '#DCDCDC',
    'grid.linewidth': 0.5,
    'grid.linestyle': '-',
    'axes.axisbelow': True,
    'axes.facecolor': 'white',
    'font.family':    'DejaVu Sans',
    'axes.titlesize': 13,
    'axes.titlepad':  13,
    'axes.labelsize': 10,
    'axes.labelpad':  8,
    'xtick.labelsize': 8,
    'ytick.labelsize': 8,
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'legend.frameon':    True,
    'legend.facecolor':  'white',
    'legend.edgecolor':  '#DCDCDC',
    'legend.framealpha': 1.0,
    'legend.fontsize':   9,
})

df_energy_weather = pd.read_csv("../data/processed/energy_weather_2019_2025.csv", parse_dates=['DateUTC'])

plt.figure(figsize=(4, 2))
sns.histplot(df_energy_weather['EnergyDemand'], bins=50, kde=True)
plt.title('Energy Demand Distribution')
plt.xlabel('Energy Demand')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# plot comparison of energy demand grouped by non-working days (holidays, weekends) and working days
df_energy_weather['DayType'] = df_energy_weather.apply(lambda row: 'non-working day' if row['is_weekend'] or row['is_holiday'] else 'working day', axis=1)

plt.figure(figsize=(4, 3))
sns.boxplot(x='DayType', y='EnergyDemand', data=df_energy_weather, hue='DayType', palette={'working day': 'steelblue', 'non-working day': 'orange'})
plt.title('Energy Demand by Day Type')
plt.xlabel('Day Type')
plt.ylabel('Energy Demand')
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

#df_energy_weather = pd.read_csv("../data/energy_weather_2019_2025.csv", index_col=False)

cols_to_plot = ['apparent_temperature', 'rain', 'snowfall', 'shortwave_radiation']
hue_col = 'is_weekend'

# plot the relationship between energy demand and apparent temperature, rain, snowfall, and shortwave radiation
fig, ax = plt.subplots(2, 2, figsize=(16, 10))
ax = ax.flatten()  # flatten the 2D array of axes for easier indexing
for i, col in enumerate(cols_to_plot): 
        sns.scatterplot(data=df_energy_weather, x=col, y='EnergyDemand', hue=hue_col, palette='Set1', ax=ax[i])
        ax[i].set_title(f'Energy Demand vs. {col.replace("_", " ").title()}')
        ax[i].set_xlabel(f'{col.replace("_", " ").title()}')
        ax[i].set_ylabel('Energy Demand (MW)')
        ax[i].grid()
        ax[i].legend(title=hue_col, loc='upper right')

plt.tight_layout()
plt.show()


In [ ]:
# normalize the weather variables and energy demand using standard scaling 
# plot energy demand and weather variables on the same plot to visualize their relationship over time

from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import pandas as pd

df_energy_weather = pd.read_csv("../data/processed/energy_weather_2019_2025.csv", parse_dates=['DateUTC'])

scaler = StandardScaler()
weather_cols = ['apparent_temperature', 'rain', 'snowfall', 'shortwave_radiation']

df_energy_weather_scaled = df_energy_weather.copy()
df_energy_weather_scaled[weather_cols + ['EnergyDemand']] = scaler.fit_transform(
    df_energy_weather[weather_cols + ['EnergyDemand']]
)

# Resample to monthly means to reduce noise
df_monthly = df_energy_weather_scaled.set_index('DateUTC')[weather_cols + ['EnergyDemand']].resample('ME').mean().reset_index()


In [ ]:
# plot energy demand, temperature and shortwave radiation on the same plot for the monthly means of the standard scaled values in 2025
df_monthly_2025 = df_monthly[df_monthly['DateUTC'].dt.year == 2025]
plt.figure(figsize=(16, 6))
plt.plot(df_monthly_2025['DateUTC'], df_monthly_2025['EnergyDemand'], label='Energy Demand', color='red')
plt.plot(df_monthly_2025['DateUTC'], df_monthly_2025['apparent_temperature'], label='Apparent Temperature', color='orange')
plt.plot(df_monthly_2025['DateUTC'], df_monthly_2025['rain'], label='Rain', color='green')
plt.plot(df_monthly_2025['DateUTC'], df_monthly_2025['snowfall'], label='Snowfall', color='blue')
plt.plot(df_monthly_2025['DateUTC'], df_monthly_2025['shortwave_radiation'], label='Shortwave Radiation', color='purple')  # visible on white
plt.title('Energy Demand and Weather Variables Over Time (Standard Scaled, Monthly Mean) in 2025')
plt.xlabel('Date')
plt.ylabel('Standard Scaled Values')
plt.legend()
plt.grid()
plt.tight_layout()
plt.show()